In [1]:
from pathlib import Path

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

In [2]:
from collections import defaultdict
import zipfile
import gc

def build_dataset(folder_name):

    # Each key is a table name (event, venue, tournament, ...)
    # Each value is a list of daily DataFrames
    tables = defaultdict(list)

    # Find all ZIP files and sort them by date
    zip_files = sorted(zip_folder.glob("*.zip"))

    # Iterate over ZIP files
    for i, zip_path in enumerate(zip_files, start=1):

        # Log the file we are working on
        print(f"{i}/{len(zip_files)}")

        # Store daily data separately for each table
        daily_tables = defaultdict(list)

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if file.endswith(".parquet") and f"/{folder_name}/" in file:

                    # Extract table name
                    filename = file.split("/")[-1].replace(".parquet", "")
                    table_name = filename.rsplit("_", 1)[0]

                    # Read parquet
                    with z.open(file) as f:
                        df = pd.read_parquet(f)

                    # Add date column
                    df["date"] = zip_path.stem

                    # Store in today's table
                    daily_tables[table_name].append(df)

        # Concat each table for this day
        for table_name, dfs in daily_tables.items():

            day_df = pd.concat(dfs, ignore_index=True)
            tables[table_name].append(day_df)

        # Free RAM
        del daily_tables
        gc.collect()

    # Concat all files
    for table_name, chunks in tables.items():
        final_df = pd.concat(chunks, ignore_index=True)
        final_df.to_parquet( zip_folder / f"{table_name}_all.parquet", index=False)

        # Get size to ensure accuracy
        print(final_df.shape)

        # Free RAM
        del final_df
        gc.collect()

    # Free RAM
    del tables
    gc.collect()

    print("Finished!")

In [ ]:
# Match tables (10 tables)
build_dataset("raw_match_parquet")

In [ ]:
# Odds
build_dataset("raw_odds_parquet")

In [ ]:
# Statistics
build_dataset("raw_statistics_parquet")

In [ ]:
# Votes
build_dataset("raw_votes_parquet")

In [ ]:
# Tennis Power
build_dataset("raw_tennis_power_parquet")

In [ ]:
# Point by Point
build_dataset("raw_point_by_point_parquet")

In [ ]:
##############################

In [4]:
#Question 4

# Read proper table (MatchTimeInfo)
import pandas as pd
MatchTimeInfo_df = pd.read_parquet(zip_folder / "time_all.parquet")

# Make a copy and clean data
MatchTimeInfo_clean = MatchTimeInfo_df.copy()

# Drop duplicates because the files are snapshots
MatchTimeInfo_clean = (
    MatchTimeInfo_clean
    .sort_values("date")
    .drop_duplicates(subset="match_id", keep="last")
)

# Drop period_4 and period_5 columns because they are totally empty
MatchTimeInfo_clean.drop(
    columns=["period_4", "period_5"],
    inplace=True
)

# Drop negative values (only 2 records are negative in period_2 column)
MatchTimeInfo_clean = MatchTimeInfo_clean[
    MatchTimeInfo_clean["period_2"] >= 0
].copy()

In [48]:
MatchTimeInfo_clean["match_duration"] = (
    MatchTimeInfo_clean["period_1"].fillna(0)
    + MatchTimeInfo_clean["period_2"].fillna(0)
    + MatchTimeInfo_clean["period_3"].fillna(0)
)

MatchTimeInfo_clean.describe()

,match_id,period_1,period_2,period_3,current_period_start_timestamp,match_duration
count,1.106900e+04,11069.000000,11069.000000,3391.000000,1.106900e+04,11069.000000
mean,1.211732e+07,2865.072545,3086.213208,3175.481569,1.709431e+09,6924.097931
std,5.353092e+04,4889.046398,5138.830977,6262.163425,1.471047e+06,9189.492166
min,1.199844e+07,2.000000,2.000000,0.000000,1.706705e+09,5.000000
25%,1.207208e+07,2043.000000,2116.000000,1957.500000,1.708236e+09,4647.000000
50%,1.212111e+07,2516.000000,2606.000000,2602.000000,1.709556e+09,5827.000000
75%,1.216095e+07,3148.000000,3255.000000,3292.000000,1.710687e+09,7539.000000
max,1.221380e+07,172606.000000,169438.000000,152072.000000,1.712055e+09,336790.000000


In [46]:
MatchTime_without_outliers = MatchTimeInfo_clean.copy()
def remove_outliers_iqr(df, columns):

    df = df.copy()

    for col in columns:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df = df[
            df[col].isna() |
            ((df[col] >= lower) & (df[col] <= upper))
        ]

    return df

MatchTime_without_outliers = remove_outliers_iqr(
    MatchTime_without_outliers,
    ["period_1", "period_2", "period_3"]
)


MatchTime_without_outliers["match_duration"] = (
    MatchTime_without_outliers["period_1"].fillna(0)
    + MatchTime_without_outliers["period_2"].fillna(0)
    + MatchTime_without_outliers["period_3"].fillna(0)
)

MatchTime_without_outliers.sort_values("match_duration", ascending=False).head()

,match_id,period_1,period_2,period_3,current_period_start_timestamp,date,match_duration
919,12024245,4564.0,3502.0,5138.0,1.706961e+09,20240203,13204.0
3749,12046963,4428.0,4763.0,3723.0,1.707328e+09,20240208,12914.0
7874,12072523,4045.0,3946.0,4771.0,1.708097e+09,20240217,12762.0
29465,12171560,4082.0,4871.0,3647.0,1.710874e+09,20240320,12600.0
14901,12088096,3756.0,4651.0,4168.0,1.709121e+09,20240228,12575.0


In [49]:
MatchTimeInfo_clean.nlargest(
    20,
    "match_duration"
)[
    ["match_id", "match_duration", "period_1", "period_2", "period_3"]
]

,match_id,match_duration,period_1,period_2,period_3
7435,12063611,336790.0,167352.0,169438.0,NaN
7434,12063587,320230.0,159144.0,161086.0,NaN
32139,12185562,177131.0,4163.0,84588.0,88380.0
17746,12121829,173416.0,172606.0,810.0,NaN
4372,12054403,163883.0,2170.0,80162.0,81551.0
7429,12063588,163033.0,158918.0,3061.0,1054.0
7425,12063582,162344.0,157690.0,4654.0,NaN
7428,12063589,162191.0,159242.0,2155.0,794.0
7433,12063615,161806.0,158909.0,2897.0,NaN
4322,12053865,160825.0,76450.0,2620.0,81755.0


In [12]:
MatchAwayTeamInfo_df = pd.read_parquet(
    zip_folder / "away_team_all.parquet"
)
MatchAwayTeamInfo_df[MatchAwayTeamInfo_df["match_id"]==12049648]

,match_id,name,slug,gender,user_count,residence,birthplace,height,weight,plays,turned_pro,current_prize,total_prize,player_id,current_rank,name_code,country,full_name,date
5417,12049648,Griekspoor T.,griekspoor-tallon,M,7204,"Nieuw Vennep, Netherlands","Haarlem, Netherlands",1.88,82.0,right-handed,2015,196223.0,2871926.0,122368,29.0,GRI,Netherlands,"Griekspoor, Tallon",20240214
5519,12049648,Griekspoor T.,griekspoor-tallon,M,7207,"Nieuw Vennep, Netherlands","Haarlem, Netherlands",1.88,82.0,right-handed,2015,196223.0,2871926.0,122368,28.0,GRI,Netherlands,"Griekspoor, Tallon",20240215
5925,12049648,Griekspoor T.,griekspoor-tallon,M,7207,"Nieuw Vennep, Netherlands","Haarlem, Netherlands",1.88,82.0,right-handed,2015,196223.0,2871926.0,122368,28.0,GRI,Netherlands,"Griekspoor, Tallon",20240216


In [13]:
MatchHomeTeamInfo_df = pd.read_parquet(
    zip_folder / "home_team_all.parquet"
)
MatchHomeTeamInfo_df[MatchHomeTeamInfo_df["match_id"]==12049648]

,match_id,name,slug,gender,user_count,residence,birthplace,height,weight,plays,turned_pro,current_prize,total_prize,player_id,current_rank,name_code,country,full_name,date
5522,12049648,Hurkacz H.,hurkacz-hubert,M,31173,"Wroclaw, Poland","Wroclaw, Poland",1.96,81.0,right-handed,2015,689771.0,12051539.0,158896,8.0,HUR,Poland,"Hurkacz, Hubert",20240214
5625,12049648,Hurkacz H.,hurkacz-hubert,M,31214,"Wroclaw, Poland","Wroclaw, Poland",1.96,81.0,right-handed,2015,689771.0,12051539.0,158896,8.0,HUR,Poland,"Hurkacz, Hubert",20240215
6030,12049648,Hurkacz H.,hurkacz-hubert,M,31214,"Wroclaw, Poland","Wroclaw, Poland",1.96,81.0,right-handed,2015,689771.0,12051539.0,158896,8.0,HUR,Poland,"Hurkacz, Hubert",20240216
